# 01 · Collect group 4 batches

Run the request cell once per batch, up to six batches for exploration. Each call can advance the shared API. Responses are saved to `data` and are excluded from Git. Never run the reset endpoint casually. The final Airflow system must process batches incrementally, one per DAG run.

## Imports

In [3]:
from pathlib import Path
import os
from datetime import datetime, timezone
import json
import requests


## Configuration

In [25]:
BASE_URL = os.environ['DATA_API_URL'].rstrip('/')
GROUP = int(os.environ['DATA_API_GROUP'])
DATA = Path('../data')
DATA.mkdir(parents=True, exist_ok=True)
print('Saved files:', [p.name for p in sorted(DATA.glob(f'group-{GROUP}-*.json'))])


Saved files: []


## Fetch one batch

Change `FETCH_ONE_BATCH` to `True` only when ready. Run the cell **once**, then set it back to `False`. No automatic retries: a failed or interrupted request may already have advanced the source. Check before trying again. The full response is saved before it is parsed, so a parsing failure does not discard it.

In [31]:
FETCH_ONE_BATCH = False

if FETCH_ONE_BATCH:
    saved = sorted(DATA.glob(f'group-{GROUP}-*.json'))
    if len(saved) >= 6:
        raise RuntimeError('6 response files already exist. Review them before requesting more.')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    path = DATA / f'group-{GROUP}-{stamp}.json'
    print('Requesting one batch; please wait...')
    try:
        with requests.get(f'{BASE_URL}/data', params={'group_number': GROUP}, timeout=(10, 180), stream=True) as response:
            response.raise_for_status()
            with path.open('xb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
    except Exception:
        print('Request or download failed. The API may have advanced; inspect any saved file and source state before retrying.')
        raise
    print('Response saved:', path)
    with path.open() as f:
        payload = json.load(f)
    print('Group:', payload.get('group_number'), '| Batch:', payload.get('batch_number'))
    print('Records:', len(payload.get('data', [])))
    if payload.get('group_number') != GROUP or not isinstance(payload.get('data'), list):
        print('Unexpected response: inspect the saved file before collecting another batch.')
else:
    print('No request made. Set FETCH_ONE_BATCH = True when ready.')


Requesting one batch; please wait...
Response saved: ../data/group-4-20261123T000142976509Z.json
Group: 4 | Batch: 5
Records: 361457
